In [1]:
from pyspark.sql import SparkSession
import os as os

OIDC_TOKEN = os.environ["OIDC_TOKEN"]
spark = (SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002")
    .config("spark.sql.catalog.westmids.scope", "openid")
    .config("spark.sql.catalog.westmids.header.X-Iceberg-Access-Delegation", "vended-credentials")
    .config("spark.sql.catalog.westmids", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.westmids.type", "rest")
    .config("spark.sql.catalog.westmids.uri", "https://westmidspolarisazure.click/api/catalog")
    .config("spark.sql.catalog.westmids.cache-enabled", "false")
    .config("spark.sql.catalog.westmids.warehouse", "azure")
    .config("spark.sql.catalog.westmids.token", OIDC_TOKEN)
    .config("spark.sql.catalog.westmids.oauth2-server-uri", "https://emrysoauth.click/realms/polaris/protocol/openid-connect/token")

     .config("spark.sql.catalog.tvs.scope", "openid")
     .config("spark.sql.catalog.tvs.header.X-Iceberg-Access-Delegation", "vended-credentials")
     .config("spark.sql.catalog.tvs", "org.apache.iceberg.spark.SparkCatalog")
     .config("spark.sql.catalog.tvs.type", "rest")
     .config("spark.sql.catalog.tvs.uri", "https://westmidspolarisazure.click/api/catalog")
     .config("spark.sql.catalog.tvs.cache-enabled", "false")
     .config("spark.sql.catalog.tvs.warehouse", "azure")
     .config("spark.sql.catalog.tvs.token", OIDC_TOKEN)
     .config("spark.sql.catalog.tvs.oauth2-server-uri", "https://tvspolarisazure.click/realms/polaris/protocol/openid-connect/token")

     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
     .config("spark.driver.allowMultipleContexts", "true")
     .config("spark.sql.iceberg.planning.preserve-data-grouping", "true")
     .config("spark.sql.sources.v2.bucketing.enabled", "true")
     .config("spark.sql.sources.bucketing.enabled", "true")
).getOrCreate()

print("Finished initializing spark session")


Finished initializing spark session


/opt/conda/lib/python3.11/site-packages/pyspark/sql/connect/conf.py:64: UserWarning: Failed to set spark.sql.extensions to Some(org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions) due to [CANNOT_MODIFY_CONFIG] Cannot modify the value of the Spark config: "spark.sql.extensions".
See also 'https://spark.apache.org/docs/latest/sql-migration-guide.html#ddl-statements'. SQLSTATE: 46110
  warnings.warn(warn)


In [2]:
df_test = spark.createDataFrame([["test", 1]], ['name', 'num'])
print("Initialized test data")

Initialized test data


In [3]:
from datetime import datetime

In [12]:
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
az_namespace = "westmids.namespaceph"
westmids_table = f"{az_namespace}.smoketesttable_" + discriminator
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {az_namespace}")
print(f"Created namespace {az_namespace}")
print("Creating table")
spark.sql(f"CREATE TABLE IF NOT EXISTS {westmids_table} (name String, num Integer) USING ICEBERG")
print(f"Writing to ${westmids_table} ...")
df_test.writeTo(westmids_table).append()
print(f"Wrote {westmids_table} to Azure. See it by running:")
print(f"az storage blob list --account-name westmidsstorageaccount  --container-name afonstorage --output table --prefix namespaceph/ --delimiter /")

Created namespace westmids.namespaceph
Creating table
Writing to $westmids.namespaceph.smoketesttable_03September2026_13_20 ...
Wrote westmids.namespaceph.smoketesttable_03September2026_13_20 to Azure. See it by running:
az storage blob list --account-name westmidsstorageaccount  --container-name afonstorage --output table --prefix namespaceph/ --delimiter /


In [5]:
spark.read.table("westmids.namespaceph.test_table").show()

+------+
| value|
+------+
| peach|
| peach|
| peach|
| peach|
| apple|
|banana|
|cherry|
+------+



In [6]:
df = spark.createDataFrame([["peach"]], ['value'])
df.writeTo("westmids.namespaceph.test_table").append()
print("done")

done


In [10]:
#
# AZURE
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
az_namespace = "azure.acceptancetests"
az_fqn = f"{az_namespace}.smoketesttable_" + discriminator

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {az_namespace}")
print(f"Created namespace {az_namespace}")

spark.sql(f"CREATE TABLE IF NOT EXISTS {az_fqn} (name String, num Integer) USING iceberg")
df_test.writeTo(az_fqn).append()

df_read = spark.read.table(az_fqn)
df_read.show()

Created namespace azure.acceptancetests
+----+---+
|name|num|
+----+---+
|test|  1|
+----+---+



In [13]:
from pyspark.sql import functions as F

westmids = spark.read.table(westmids_table)
tvs = spark.read.table(az_fqn)

westmids.join(tvs, on="num", how="inner").show()

+---+----+----+
|num|name|name|
+---+----+----+
|  1|test|test|
+---+----+----+



In [16]:
westmids.select(F.input_file_name().alias("file")).distinct().show(truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file                                                                                                                                                                              |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://afonstorage@westmidsstorageaccount.dfs.core.windows.net/namespaceph/smoketesttable_03September2026_13_20/data/00000-8-ba3b86c8-b7c2-48b5-9af0-8593f29214d4-0-00001.parquet|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+



In [17]:
tvs.select(F.input_file_name().alias("file")).distinct().show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file                                                                                                                                                                                 |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://afonstorage@tvsafonstorageaccount.dfs.core.windows.net/acceptancetests/smoketesttable_03September2026_12_27/data/00000-4-b6e5909c-5f56-46dc-bc76-a1516e4aafcc-0-00001.parquet|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+

